In [3]:
import pandas as pd
from pathlib import Path

# =========================================================
# LDBC SNB CROSS-SCALE COMPARISON — CORRECTED VERSION
# =========================================================
# Aligned with comparison_sfs_imdb.ipynb:
#
# 1) Reads one corrected per-scale analysis file per result folder.
# 2) Filters explicitly by run_phase.
# 3) Computes cross-scale summary.
# 4) Includes near-best preservation when available.
# 5) Exports final cross-scale artifacts.
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------

base_results = Path(
    "/home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results"
)

paths = {
    "sf0.1": base_results / "ldbc_snb_sf0_1_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
    "sf1":   base_results / "ldbc_snb_sf1_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
    "sf3":   base_results / "ldbc_snb_sf3_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
}

fallback_filename = "schemalens_reduction_analysis_hot.csv"

selected_run_phase = "hot"

# ---------------------------------------------------------
# 2) Read all per-scale corrected analysis files
# ---------------------------------------------------------

dfs = []

for scale, path in paths.items():
    if not path.exists():
        fallback_path = path.parent.parent / fallback_filename
        if fallback_path.exists():
            path = fallback_path
        else:
            raise FileNotFoundError(
                f"Could not find analysis file for {scale}.\n"
                f"Tried:\n- {path}\n- {fallback_path}"
            )

    print(f"Reading {scale}: {path}")

    df = pd.read_csv(path)

    if "scale_label" not in df.columns:
        df["scale_label"] = scale

    # Force expected scale label when file has one scale only
    if df["scale_label"].nunique() == 1:
        df["scale_label"] = scale

    if "run_phase" in df.columns:
        df = df[df["run_phase"] == selected_run_phase].copy()
    else:
        df["run_phase"] = selected_run_phase

    dfs.append(df)

cross_scale_schemalens_df = pd.concat(dfs, ignore_index=True)

# ---------------------------------------------------------
# 3) Main cross-scale summary
# ---------------------------------------------------------

agg_dict = dict(
    n_queries=("query_name", "nunique"),
    avg_DSR=("DSR", "mean"),
    top1_preservation=("top1_preserved_by_activated", "mean"),
    mean_activated_regret=("activated_regret", "mean"),
    mean_primary_regret=("primary_regret", lambda x: x.dropna().mean()),
    primary_winners=("best_group", lambda x: int((x == "primary").sum())),
    secondary_winners=("best_group", lambda x: int((x == "secondary_affected").sum())),
    control_winners=("best_group", lambda x: int((x == "control").sum())),
)

if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    agg_dict["near_best_preservation"] = ("near_best_preserved_by_activated", "mean")

summary_df = (
    cross_scale_schemalens_df
    .groupby("scale_label")
    .agg(**agg_dict)
    .reset_index()
)

scale_order = {"sf0.1": 0, "sf1": 1, "sf3": 2, "sf10": 3}
summary_df["scale_order"] = summary_df["scale_label"].map(scale_order).fillna(999)
summary_df = summary_df.sort_values("scale_order").drop(columns=["scale_order"])

display(summary_df)

# ---------------------------------------------------------
# 4) Secondary winners table
# ---------------------------------------------------------

secondary_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    secondary_cols.insert(1, "run_phase")

secondary_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "secondary_affected"
][secondary_cols].sort_values(["scale_label", "official_id"])

display(secondary_df)

# ---------------------------------------------------------
# 5) Control winners table
# ---------------------------------------------------------

control_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    control_cols.insert(1, "run_phase")

control_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "control"
][control_cols].sort_values(["scale_label", "official_id"])

display(control_df)

# ---------------------------------------------------------
# 6) Best-by-scale overview
# ---------------------------------------------------------

best_by_scale_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_group",
    "best_design_pattern",
    "best_p95_ms",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    best_by_scale_cols.insert(1, "run_phase")

best_by_scale_df = (
    cross_scale_schemalens_df[best_by_scale_cols]
    .sort_values(["scale_label", "official_id"])
    .reset_index(drop=True)
)

display(best_by_scale_df)

# ---------------------------------------------------------
# 7) p95 pivot across scales
# ---------------------------------------------------------

p95_pivot_df = (
    cross_scale_schemalens_df
    .pivot_table(
        index=["official_id", "query_name"],
        columns="scale_label",
        values="best_p95_ms",
        aggfunc="first"
    )
    .reset_index()
)

if "sf0.1" in p95_pivot_df.columns and "sf1" in p95_pivot_df.columns:
    p95_pivot_df["sf1_vs_sf0_1_ratio"] = p95_pivot_df["sf1"] / p95_pivot_df["sf0.1"]

if "sf1" in p95_pivot_df.columns and "sf3" in p95_pivot_df.columns:
    p95_pivot_df["sf3_vs_sf1_ratio"] = p95_pivot_df["sf3"] / p95_pivot_df["sf1"]

if "sf0.1" in p95_pivot_df.columns and "sf3" in p95_pivot_df.columns:
    p95_pivot_df["sf3_vs_sf0_1_ratio"] = p95_pivot_df["sf3"] / p95_pivot_df["sf0.1"]

display(p95_pivot_df)

# ---------------------------------------------------------
# 8) Optional near-best summary
# ---------------------------------------------------------

if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    near_best_summary_df = (
        cross_scale_schemalens_df
        .groupby("scale_label")
        .agg(
            near_best_preservation=("near_best_preserved_by_activated", "mean"),
            mean_activated_regret=("activated_regret", "mean"),
        )
        .reset_index()
    )

    near_best_summary_df["scale_order"] = near_best_summary_df["scale_label"].map(scale_order).fillna(999)
    near_best_summary_df = near_best_summary_df.sort_values("scale_order").drop(columns=["scale_order"])

    display(near_best_summary_df)
else:
    near_best_summary_df = pd.DataFrame()

# ---------------------------------------------------------
# 9) Save outputs
# ---------------------------------------------------------

out_dir = base_results / "cross_scale_analysis_final_corrected"
out_dir.mkdir(parents=True, exist_ok=True)

cross_scale_schemalens_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_schemalens_analysis_final.csv",
    index=False
)

summary_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_summary_final.csv",
    index=False
)

secondary_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_secondary_winners_final.csv",
    index=False
)

control_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_control_winners_final.csv",
    index=False
)

best_by_scale_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_best_by_scale_final.csv",
    index=False
)

p95_pivot_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_best_p95_by_query_final.csv",
    index=False
)

if not near_best_summary_df.empty:
    near_best_summary_df.to_csv(
        out_dir / "ldbc_snb_cross_scale_near_best_summary_final.csv",
        index=False
    )

print("Saved to:", out_dir)

Reading sf0.1: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf0_1_full_fiben_format_clean/ldbc_snb_analysis_outputs/ldbc_snb_summary_hot_by_scale.csv
Reading sf1: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf1_full_fiben_format_clean/ldbc_snb_analysis_outputs/ldbc_snb_summary_hot_by_scale.csv
Reading sf3: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/ldbc_snb_sf3_full_fiben_format_clean/ldbc_snb_analysis_outputs/ldbc_snb_summary_hot_by_scale.csv


,scale_label,n_queries,avg_DSR,top1_preservation,mean_activated_regret,mean_primary_regret,primary_winners,secondary_winners,control_winners,near_best_preservation
0,sf0.1,22,0.713636,1.000000,0.000000,0.025313,19,3,0,1.000000
1,sf1,22,0.713636,0.954545,0.023537,0.033819,15,6,1,0.954545
2,sf3,22,0.713636,1.000000,0.000000,0.089429,15,7,0,1.000000


,scale_label,run_phase,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
4,sf0.1,hot,IC5,IC5_NewGroups,G7,containment_baseline,135.322968,G0,158.656699,0.172430
6,sf0.1,hot,IC7,IC7_RecentLikers,G4,explicit_edge_collection,7.464429,G0,10.145230,0.359143
16,sf0.1,hot,IS2,IS2_RecentMessagesOfPerson,G6,referenced_or_reverse_indexed_edges,2.255789,NaN,NaN,NaN
24,sf1,hot,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,G7,containment_baseline,191.817495,G3,196.627160,0.025074
26,sf1,hot,IC5,IC5_NewGroups,G6,referenced_or_reverse_indexed_edges,176.753622,G0,185.673138,0.050463
34,sf1,hot,INS6,INS6_AddPost,G9,hybrid_containment,2.188292,G3,2.343452,0.070905
35,sf1,hot,INS7,INS7_AddComment,G9,hybrid_containment,2.159610,G3,2.172902,0.006154
38,sf1,hot,IS2,IS2_RecentMessagesOfPerson,G9,hybrid_containment,3.230855,NaN,NaN,NaN
43,sf1,hot,IS7,IS7_RepliesOfMessage,G9,hybrid_containment,11.009079,G0,11.447059,0.039784
48,sf3,hot,IC5,IC5_NewGroups,G7,containment_baseline,193.801405,G3,205.203150,0.058832


,scale_label,run_phase,official_id,query_name,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret,activated_regret
40,sf1,hot,IS4,IS4_ContentOfMessage,G0,root_with_references,2.288074,G1,3.472862,0.51781,0.51781


,scale_label,run_phase,official_id,query_name,best_config,best_group,best_design_pattern,best_p95_ms
0,sf0.1,hot,IC1,IC1_TransitiveFriendsWithName,G3,primary,root_with_references_or_summaries,131.361695
1,sf0.1,hot,IC2,IC2_RecentMessagesByFriends,G3,primary,root_with_references_or_summaries,28.638829
2,sf0.1,hot,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,G0,primary,root_with_references,164.711398
3,sf0.1,hot,IC4,IC4_NewTopics,G3,primary,root_with_references_or_summaries,44.056135
4,sf0.1,hot,IC5,IC5_NewGroups,G7,secondary_affected,containment_baseline,135.322968
...,...,...,...,...,...,...,...,...
61,sf3,hot,IS3,IS3_FriendsOfPerson,G0,primary,root_with_references,2.661645
62,sf3,hot,IS4,IS4_ContentOfMessage,G1,primary,single_entity_lookup,1.261365
63,sf3,hot,IS5,IS5_CreatorOfMessage,G0,primary,root_with_references,1.424061
64,sf3,hot,IS6,IS6_ForumOfMessage,G0,secondary_affected,root_with_references,1.332350


scale_label,official_id,query_name,sf0.1,sf1,sf3,sf1_vs_sf0_1_ratio,sf3_vs_sf1_ratio,sf3_vs_sf0_1_ratio
0,IC1,IC1_TransitiveFriendsWithName,131.361695,227.727482,260.022248,1.733591,1.141813,1.979437
1,IC2,IC2_RecentMessagesByFriends,28.638829,37.090559,36.105321,1.295114,0.973437,1.260712
2,IC3,IC3_FriendsAndFriendsOfFriendsInCountries,164.711398,191.817495,205.565454,1.164567,1.071672,1.248034
3,IC4,IC4_NewTopics,44.056135,66.387148,45.188472,1.506876,0.680681,1.025702
4,IC5,IC5_NewGroups,135.322968,176.753622,193.801405,1.306161,1.096449,1.432140
5,IC6,IC6_TagCoOccurrence,198.168794,198.528747,253.192134,1.001816,1.275342,1.277659
6,IC7,IC7_RecentLikers,7.464429,7.009322,8.036814,0.939030,1.146589,1.076682
7,INS1,INS1_AddPerson,1.184437,1.624116,1.432566,1.371213,0.882059,1.209491
8,INS2,INS2_AddLikeToPost,1.049605,1.159713,1.630815,1.104904,1.406224,1.553742
9,INS3,INS3_AddLikeToComment,1.222271,1.868959,1.312449,1.529088,0.702235,1.073779


,scale_label,near_best_preservation,mean_activated_regret
0,sf0.1,1.000000,0.000000
1,sf1,0.954545,0.023537
2,sf3,1.000000,0.000000


Saved to: /home/jovyan/privado/framework evaluation approachs/framework with dataset LDBC SNB/results/cross_scale_analysis_final_corrected
